# Retail Sales Performance & Customer Insights Dashboard

This notebook documents the data cleaning, feature engineering, and exploratory analysis steps used to build the portfolio project around the UCI `Online Retail` dataset.

## Business Objective

Analyze transaction-level retail data to understand revenue trends, customer behavior, and product performance. The output supports stakeholder reporting in Power BI and provides actionable recommendations for commercial growth.

## Key Questions

- How does revenue change by month and season?
- Which products and derived product categories generate the most revenue?
- How much revenue comes from new versus repeat customers?
- Which customer segments deserve retention or upsell attention based on RFM analysis?

In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent
sys.path.append(str(BASE_DIR))

from src.data.build_retail_assets import (
    load_raw_data,
    clean_transactions,
    build_sales_fact,
    build_returns_fact,
    build_customer_features,
    build_product_dimension,
    build_date_dimension,
    add_date_keys,
)


In [2]:
raw_df = load_raw_data()
raw_df.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom


In [3]:
raw_df.shape, raw_df.columns.tolist()

((541909, 8),
 ['invoice_no',
  'stock_code',
  'description',
  'quantity',
  'invoice_date',
  'unit_price',
  'customer_id',
  'country'])

## Data Cleaning

In [4]:
cleaned_df, quality_report = clean_transactions(raw_df)
quality_report

{'raw_rows': 541909,
 'missing_description_rows': 1454,
 'missing_customer_rows': 135080,
 'return_rows': 10624,
 'zero_or_negative_price_rows': 2517,
 'countries': 38,
 'date_min': '2010-12-01 08:26:00',
 'date_max': '2011-12-09 12:50:00'}

In [5]:
sales_fact = build_sales_fact(cleaned_df)
returns_fact = build_returns_fact(cleaned_df)
sales_fact, customer_dim = build_customer_features(sales_fact)
sales_fact = add_date_keys(sales_fact)
returns_fact = add_date_keys(returns_fact)
product_dim = build_product_dimension(sales_fact)
date_dim = build_date_dimension(cleaned_df)

print('fact_sales:', sales_fact.shape)
print('fact_returns:', returns_fact.shape)
print('dim_customer_rfm:', customer_dim.shape)
print('dim_product:', product_dim.shape)
print('dim_date:', date_dim.shape)

fact_sales: (530104, 29)
fact_returns: (10624, 27)
dim_customer_rfm: (4338, 15)
dim_product: (3922, 5)
dim_date: (374, 10)


## KPI Snapshot

In [6]:
kpi_snapshot = {
    'total_revenue': round(float(sales_fact['line_revenue'].sum()), 2),
    'orders': int(sales_fact['invoice_no'].nunique()),
    'active_customers': int(sales_fact['customer_id'].nunique(dropna=True)),
    'average_order_value': round(float(sales_fact.groupby('invoice_no')['line_revenue'].sum().mean()), 2),
    'return_amount': round(float(returns_fact['return_amount'].sum()), 2),
}
kpi_snapshot

{'total_revenue': 10666684.54,
 'orders': 19960,
 'active_customers': 4338,
 'average_order_value': 534.4,
 'return_amount': 896812.49}

## Monthly Revenue Trend

In [7]:
monthly_revenue = (
    sales_fact.groupby('invoice_month_start', as_index=False)
    .agg(revenue=('line_revenue', 'sum'), orders=('invoice_no', 'nunique'))
    .sort_values('invoice_month_start')
)
monthly_revenue['aov'] = monthly_revenue['revenue'] / monthly_revenue['orders']
monthly_revenue

,invoice_month_start,revenue,orders,aov
0,2010-12-01,823746.140,1559,528.381103
1,2011-01-01,691364.560,1086,636.615617
2,2011-02-01,523631.890,1100,476.028991
3,2011-03-01,717639.360,1454,493.562146
4,2011-04-01,537808.621,1246,431.628107
5,2011-05-01,770536.020,1681,458.379548
6,2011-06-01,761739.900,1533,496.894912
7,2011-07-01,719221.191,1475,487.607587
8,2011-08-01,759138.380,1361,557.779853
9,2011-09-01,1058590.172,1837,576.260300


## Product and Category Performance

In [8]:
top_categories = (
    sales_fact.groupby('product_category', as_index=False)['line_revenue']
    .sum()
    .sort_values('line_revenue', ascending=False)
)
top_categories.head(10)

,product_category,line_revenue
4,Home Decor & Lighting,3234511.320
9,Storage & Organization,2193348.950
5,Kitchen & Dining,1729191.920
6,Miscellaneous,1268913.790
8,Stationery & Crafts,717911.750
3,Holiday & Seasonal,535852.030
1,Fees & Adjustments,397123.814
2,Garden & Outdoor,196631.360
7,Party & Gift,163914.480
10,Toys & Kids,119920.270


In [9]:
top_products = (
    sales_fact.loc[sales_fact['line_type'] == 'Merchandise']
    .groupby(['stock_code', 'description'], as_index=False)
    .agg(revenue=('line_revenue', 'sum'), units=('quantity', 'sum'))
    .sort_values('revenue', ascending=False)
    .head(10)
)
top_products

,stock_code,description,revenue,units
1340,22423,REGENCY CAKESTAND 3 TIER,174484.74,13879
2663,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995
3635,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104340.29,37599
2872,47566,PARTY BUNTING,99504.33,18295
3614,85099B,JUMBO BAG RED RETROSPOT,94340.05,48474
2119,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033
2026,23084,RABBIT NIGHT LIGHT,66964.99,30788
1022,22086,PAPER CHAIN KIT 50'S CHRISTMAS,64952.29,19355
3411,84879,ASSORTED COLOUR BIRD ORNAMENT,59094.93,36461
3054,79321,CHILLI LIGHTS,54117.76,10306


## Customer Segmentation (RFM)

In [10]:
rfm_distribution = (
    customer_dim.groupby('rfm_segment', as_index=False)
    .agg(customers=('customer_id', 'nunique'), revenue=('monetary_value', 'sum'))
    .sort_values('revenue', ascending=False)
)
rfm_distribution

,rfm_segment,customers,revenue
2,Champions,945,5743052.840
4,Loyal Customers,506,941247.071
0,At Risk,465,763985.661
3,Hibernating,1075,525138.312
7,Potential Loyalists,390,485586.600
5,Need Attention,444,170798.390
1,Big Spenders,92,164399.390
8,Promising,379,100894.700
6,New Customers,42,16304.940


In [11]:
new_vs_repeat = (
    sales_fact.loc[sales_fact['customer_order_type'].isin(['New', 'Repeat'])]
    .groupby('customer_order_type', as_index=False)['line_revenue']
    .sum()
)
new_vs_repeat

,customer_order_type,line_revenue
0,New,2253233.583
1,Repeat,6658174.321


## Files Created by the Pipeline

The supporting Python pipeline writes clean datasets, dimensional tables, SQL-ready SQLite tables, and SVG report assets to the repository. Those outputs are used by the README and Power BI build instructions.

In [12]:
summary_path = BASE_DIR / 'data' / 'processed' / 'analytics_summary.json'
summary = json.loads(summary_path.read_text())
summary['kpis']

{'total_revenue': 10666684.54,
 'total_orders': 19960,
 'active_customers': 4338,
 'average_order_value': 534.4,
 'return_amount': 896812.49,
 'return_rows': 10624}

## Conclusion

The dataset shows strong Q4 seasonality, a heavy concentration of revenue in the United Kingdom, and a meaningful dependence on repeat customers and high-value RFM segments. These outputs provide a strong foundation for stakeholder storytelling inside Power BI.